# CS521 VideoChef — Results Analysis

This notebook loads the evaluation results from `results/combined_results.csv` and reproduces the key figures from the proposal:

1. **Speedup comparison** across VideoChef variants, IRA baseline, and exact execution
2. **Quality (PSNR / SSIM) vs threshold violation rate** per configuration
3. **Error model F-1 score** comparison (Model-C vs CA vs CAD)
4. **Key-frame strategy tradeoff** (fixed interval vs scene change)
5. **PSNR vs SSIM as quality metric** — do they lead to different approximation choices?
6. **Canary quality vs full-frame quality** scatter (reproduces Figure 2 of the paper)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

import config

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

RESULTS_DIR = Path('../results')
MODELS_DIR  = Path('../models')

In [ ]:
csv_path = RESULTS_DIR / 'combined_results.csv'
if not csv_path.exists():
    print(f'Results file not found: {csv_path}')
    print('Run  python evaluate.py  first.')
else:
    df = pd.read_csv(csv_path)
    df['speedup']        = pd.to_numeric(df['speedup'],        errors='coerce')
    df['mean_psnr']      = pd.to_numeric(df['mean_psnr'],      errors='coerce')
    df['mean_ssim']      = pd.to_numeric(df['mean_ssim'],      errors='coerce')
    df['violation_rate'] = pd.to_numeric(df['violation_rate'], errors='coerce')
    print(f'Loaded {len(df)} rows from {csv_path}')
    print(f'Unique configs: {sorted(df["config"].unique())}')
    df.head()

## Figure 1 — Mean Speedup by Configuration

In [ ]:
summary = (df.groupby('config')[['speedup', 'mean_psnr', 'mean_ssim', 'violation_rate']]
             .mean()
             .sort_values('speedup', ascending=False)
             .reset_index())

fig, ax = plt.subplots(figsize=(13, 5))
colors  = sns.color_palette('tab10', len(summary))
bars    = ax.barh(summary['config'], summary['speedup'], color=colors)
ax.axvline(1.0, color='black', linewidth=1, linestyle='--', label='Exact (1×)')
ax.set_xlabel('Speedup relative to exact')
ax.set_title('Mean Speedup Across Test Videos')
ax.legend()
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'fig_speedup.pdf'), bbox_inches='tight')
plt.show()

## Figure 2 — Speedup vs Quality Trade-off

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_col, y_label, threshold in [
    (axes[0], 'mean_psnr', 'Mean PSNR (dB)', config.DEFAULT_PSNR_THRESHOLD),
    (axes[1], 'mean_ssim', 'Mean SSIM',       config.SSIM_THRESHOLD),
]:
    sub = summary.dropna(subset=[y_col])
    sub = sub[~sub[y_col].isin([float('inf')])]
    ax.scatter(sub['speedup'], sub[y_col],
               s=80, alpha=0.8, zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row['config'], (row['speedup'], row[y_col]),
                    fontsize=7, xytext=(4, 2), textcoords='offset points')
    ax.axhline(threshold, color='red', linestyle='--',
               label=f'Threshold = {threshold}')
    ax.set_xlabel('Speedup')
    ax.set_ylabel(y_label)
    ax.set_title(f'Speedup vs {y_label}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'fig_speedup_quality.pdf'), bbox_inches='tight')
plt.show()

## Figure 3 — Error Model Comparison (VideoChef variants)

In [ ]:
vc_df = df[df['config'].str.startswith('VC-')].copy()
vc_df['model_label'] = vc_df['model'].map({'C': 'Model-C', 'CA': 'Model-CA', 'CAD': 'Model-CAD'})

if not vc_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    metrics   = [('speedup', 'Speedup'),
                 ('mean_psnr', 'Mean PSNR (dB)'),
                 ('violation_rate', 'Violation Rate')]
    for ax, (col, label) in zip(axes, metrics):
        order = ['Model-C', 'Model-CA', 'Model-CAD']
        sub   = vc_df.groupby('model_label')[col].mean().reindex(order)
        ax.bar(sub.index, sub.values,
               color=sns.color_palette('Blues_d', 3))
        ax.set_title(label)
        ax.set_xlabel('Error Model')
        if col == 'mean_psnr':
            ax.axhline(config.DEFAULT_PSNR_THRESHOLD, color='red',
                       linestyle='--', label='30 dB threshold')
            ax.legend(fontsize=9)
    plt.suptitle('Error Model Comparison')
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'fig_model_comparison.pdf'), bbox_inches='tight')
    plt.show()
else:
    print('No VideoChef rows found – run evaluate.py first.')

## Figure 4 — Key-frame Strategy Comparison

In [ ]:
if not vc_df.empty:
    kf_summary = vc_df.groupby('keyframe')[['speedup', 'violation_rate', 'n_keyframes']].mean()
    print(kf_summary.to_string())

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, (col, label) in zip(axes, [
        ('speedup',        'Speedup'),
        ('violation_rate', 'Violation Rate'),
        ('n_keyframes',    'Avg. Keyframes'),
    ]):
        data = kf_summary[col].sort_values(ascending=False)
        ax.bar(data.index, data.values,
               color=sns.color_palette('Greens_d', len(data)))
        ax.set_title(label)
        ax.set_xlabel('Keyframe Strategy')
    plt.suptitle('Key-frame Strategy Comparison')
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'fig_keyframe_comparison.pdf'), bbox_inches='tight')
    plt.show()

## Figure 5 — PSNR vs SSIM as Quality Metric

In [ ]:
if not vc_df.empty:
    metric_summary = vc_df.groupby('metric')[['speedup', 'violation_rate', 'mean_ssim']].mean()
    print(metric_summary.to_string())

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, (col, label) in zip(axes, [
        ('speedup',        'Mean Speedup'),
        ('violation_rate', 'PSNR Violation Rate'),
        ('mean_ssim',      'Mean SSIM'),
    ]):
        data = metric_summary[col]
        ax.bar(data.index, data.values,
               color=sns.color_palette('Oranges_d', len(data)))
        ax.set_title(label)
        ax.set_xlabel('Quality Metric Used for Approximation')
    plt.suptitle('PSNR vs SSIM as Approximation Quality Metric')
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'fig_metric_comparison.pdf'), bbox_inches='tight')
    plt.show()

## Figure 6 — Canary Quality vs Full-frame Quality Scatter

Reproduces the key Figure 2 from the VideoChef paper showing that canary PSNR
is systematically *lower* than full-frame PSNR for the same approximation settings.

In [ ]:
import itertools
import random

from src.canary  import generate_canary
from src.filters import apply_pipeline
from src.quality import compute_psnr
from src.video_io import read_frames

PIPELINE_NAME = 'BVD'
pipeline      = config.PIPELINES[PIPELINE_NAME]
n_filters     = len(pipeline)
exact_al      = [1] * n_filters

video_files = sorted(Path('../data/videos').rglob('*.mp4'))
if not video_files:
    print('No videos found. Run  python scripts/generate_test_videos.py  first.')
else:
    vp     = video_files[0]
    frames, _ = read_frames(str(vp), max_frames=5)
    all_combos = list(itertools.product(range(1, config.MAX_AL + 1), repeat=n_filters))
    combos = random.sample(all_combos, min(30, len(all_combos)))

    C_pts, F_pts = [], []
    for frame in frames[:3]:
        canary       = generate_canary(frame, config.CANARY_SCALE)
        canary_exact = apply_pipeline(canary, pipeline, exact_al)
        full_exact   = apply_pipeline(frame,  pipeline, exact_al)
        for combo in combos:
            al    = list(combo)
            C_val = compute_psnr(canary_exact, apply_pipeline(canary, pipeline, al))
            F_val = compute_psnr(full_exact,   apply_pipeline(frame,  pipeline, al))
            if np.isfinite(C_val) and np.isfinite(F_val):
                C_pts.append(C_val)
                F_pts.append(F_val)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(C_pts, F_pts, alpha=0.5, s=20, label='Approximation settings')
    lim = [min(C_pts + F_pts) - 1, max(C_pts + F_pts) + 1]
    ax.plot(lim, lim, 'r--', label='F = C (IRA assumption)')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Canary PSNR  C  (dB)')
    ax.set_ylabel('Full-frame PSNR  F  (dB)')
    ax.set_title(f'Canary vs Full-frame Quality  –  pipeline {PIPELINE_NAME}')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'fig_canary_vs_full.pdf'), bbox_inches='tight')
    plt.show()
    pct_above = 100.0 * np.mean(np.array(F_pts) > np.array(C_pts))
    print(f'{pct_above:.1f}% of points have F > C  '
          f'(paper reports ~98 %, validating IRA under-estimates quality)')

## Table 1 — F-1 Score of Error Models

In [ ]:
from src.error_model import load_model, evaluate_model_f1, IRABaseline

model_results = []

for pipeline_name in config.PIPELINES:
    for metric in ['psnr', 'ssim']:
        threshold = (config.DEFAULT_PSNR_THRESHOLD if metric == 'psnr'
                     else config.SSIM_THRESHOLD)
        for mtype in ['C', 'CA', 'CAD']:
            path = MODELS_DIR / f'model_{pipeline_name}_{mtype}_{metric}.pkl'
            if not path.exists():
                continue
            try:
                model = load_model(str(path))
                # We don't have held-out data here – note for full evaluation
                # that train.py prints F-1 on training data as a sanity check.
                model_results.append({'pipeline': pipeline_name,
                                      'metric': metric,
                                      'model': mtype,
                                      'fitted': getattr(model, 'fitted', False)})
            except Exception as e:
                print(f'Could not load {path.name}: {e}')

if model_results:
    pd.DataFrame(model_results).to_string(index=False)
    print(pd.DataFrame(model_results).to_string(index=False))
else:
    print('No trained models found.  Run  python train.py  first.')

## Table 2 — Violation Rate Summary

In [ ]:
if 'df' in dir():
    viol_table = (df.pivot_table(
        index='config',
        values='violation_rate',
        aggfunc='mean'
    ).sort_values('violation_rate'))
    print('Mean PSNR Violation Rate (fraction of frames < threshold):')
    print(viol_table.to_string())